# RAG pipeline alapjai — Jeff Thompson receptgyűjteményén

**Házi feladat (2. hét, AI Engineering alapképzés)**

## 1. Bevezetés

Ez a notebook egy RAG (Retrieval-Augmented Generation) rendszer alapjait építi fel
[Jeff Thompson markdown receptgyűjteményén](https://github.com/jeffThompson/Recipes)
(85 recept, egységes-ish markdown struktúra). A cél:

1. Az adat struktúrájának megismerése
2. Legalább 3 chunking stratégia kipróbálása és összehasonlítása
3. Metadata kinyerése chunkonként
4. Vektoradatbázis (Qdrant) feltöltése
5. Alapvető keresési funkciók (szemantikus, hozzávaló-alapú, metadata-szűrés)
6. Tanulságok összefoglalása

A repót a `work/recipes-src/` mappába klónoztuk (`git clone --depth 1`).

## 2. Adatelemzés

Először megnézzük, mennyire egységes a recept-fájlok struktúrája — ez dönti el,
hogy a szekció-alapú (B) chunking stratégia mennyire lesz megbízható.

In [1]:
import glob
import re
import collections
from pathlib import Path

# Robusztus útvonal-keresés: nem tételezzük fel, honnan indítják a kernelt
# (docs/hw2-ből, a repo gyökeréből, vagy VS Code máshonnan nyitja meg) —
# több jelölt helyen keressük a receptmappát, és az elsőt vesszük, ami tényleg
# létezik és tartalmaz .md fájlokat.
_candidates = [
    Path("work/recipes-src/recipes"),           # docs/hw2-ből indítva
    Path("docs/hw2/work/recipes-src/recipes"),   # repo gyökeréből indítva
    Path.cwd() / "work/recipes-src/recipes",     # explicit cwd-hez képest
]

RECIPES_DIR = next((p for p in _candidates if p.exists() and any(p.glob("*.md"))), None)

if RECIPES_DIR is None:
    raise FileNotFoundError(
        "Nem találom a receptmappát egyik várható helyen sem: "
        + ", ".join(str(p.resolve()) for p in _candidates)
        + ". Ellenőrizd, hogy a 'work/recipes-src' repó klónozva van-e, "
          "és hogy a Jupyter kernel munkakönyvtára a docs/hw2 mappa-e "
          "(VS Code-ban: jobb klikk a notebookra → 'Select Kernel', "
          "vagy futtasd le a %pwd magic parancsot az ellenőrzéshez)."
    )

print(f"RECIPES_DIR feloldva: {RECIPES_DIR.resolve()}")

files = sorted(RECIPES_DIR.glob("*.md"))
print(f"Talált receptfájlok száma: {len(files)}")
print("Első 5 fájl:", [f.name for f in files[:5]])

RECIPES_DIR feloldva: C:\Users\Laci\projects\rezsiradar\docs\hw2\work\recipes-src\recipes
Talált receptfájlok száma: 85
Első 5 fájl: ['aloo-matar.md', 'austro-hungarian-cream-of-horseradish-soup.md', 'baba-ganoush.md', 'bagels.md', 'barbeque-sauce.md']


In [2]:
# Egy konkrét recept nyers tartalma — hogy lássuk a valódi struktúrát
sample = (RECIPES_DIR / "beef-tacos.md").read_text(encoding="utf-8")
print(sample)

# Beef Tacos  

## info  
* About 15 minutes  
* 2 servings  

## ingredients
* 1 lbs ground beef  
* 1 tsp ground cumin  
* 1 tsp smoked paprika  
* 1/2 tsp garlic powder  
* 1/2 tsp chili powder  
* 1/2 tsp salt  
* 1/2 tsp black pepper  
* 1 tbsp water  

## steps  
1. Heat oil in a cast iron skillet over medium heat, add the beef and cook until it starts to brown  
2. Add seasonings, stir, and cook until done  
3. Add 1 tbsp water to make the beef a bit saucier, stir to combine  

## based on  
* https://www.howsweeteats.com/2019/01/ground-beef-tacos  




A feladatleírás példája csak `ingredients` / `steps` / `notes` szekciókat mutat —
a valódi fájlokban ennél több van: egy `## info` szekció (elkészítési idő, adagszám)
és egy `## based on` szekció (forrás-linkek). Nézzük meg szisztematikusan, melyik
recept milyen szekciókat tartalmaz.

In [3]:
heading_sets = collections.Counter()
heading_freq = collections.Counter()
lengths = []
subheader_count = 0

for fn in files:
    text = fn.read_text(encoding="utf-8")
    lengths.append(len(text))

    headings = tuple(re.findall(r"^##\s+(.+?)\s*$", text, re.M))
    heading_sets[headings] += 1
    for h in headings:
        heading_freq[h.lower()] += 1

    lines = text.split("\n")
    if len(lines) > 1 and lines[0].startswith("# ") and lines[1].strip() and not lines[1].startswith("#"):
        subheader_count += 1

print(f"Vizsgált fájlok: {len(files)}")

Vizsgált fájlok: 85


In [4]:
print("Szekció-kombinációk (hányféle struktúra fordul elő):\n")
for hs, cnt in heading_sets.most_common():
    print(f"{cnt:3d}x  {hs}")

Szekció-kombinációk (hányféle struktúra fordul elő):

 32x  ('info', 'ingredients', 'steps', 'notes', 'based on')
 25x  ('ingredients', 'steps', 'notes', 'based on')
 15x  ('ingredients', 'steps', 'based on')
  7x  ('info', 'ingredients', 'steps', 'based on')
  2x  ('ingredients', 'steps', 'notes:', 'based on:')
  1x  ('info', 'ingredients', 'steps', 'notes', 'based on:')
  1x  ('info', 'ingredients', 'steps', 'based on:')
  1x  ('ingredients', 'steps', 'notes', 'based on:')
  1x  ('ingredients', 'steps', 'based on:')


In [5]:
print("Egyes szekciócímek gyakorisága (hány receptben szerepel):\n")
for h, cnt in heading_freq.most_common():
    pct = cnt / len(files) * 100
    print(f"{cnt:3d}/{len(files)} ({pct:5.1f}%)  {h!r}")

Egyes szekciócímek gyakorisága (hány receptben szerepel):

 85/85 (100.0%)  'ingredients'
 85/85 (100.0%)  'steps'
 79/85 ( 92.9%)  'based on'
 59/85 ( 69.4%)  'notes'
 41/85 ( 48.2%)  'info'
  6/85 (  7.1%)  'based on:'
  2/85 (  2.4%)  'notes:'


In [6]:
print(f"Fájlhossz (karakter): min={min(lengths)}, max={max(lengths)}, "
      f"átlag={sum(lengths)//len(lengths)}")
print(f"Alcímes receptek (cím alatt egy szöveg-sor, mielőtt szekció jönne): "
      f"{subheader_count}/{len(files)}")

Fájlhossz (karakter): min=459, max=4547, átlag=1517
Alcímes receptek (cím alatt egy szöveg-sor, mielőtt szekció jönne): 22/85


### Megfigyelések

- **`ingredients` és `steps` 100%-ban jelen van** minden receptben — ezekre
  biztonságosan lehet szekció-alapú chunkinget építeni.
- **`info`** (idő/adag) csak a receptek **~48%**-ában van meg, **`notes`** ~69%-ban —
  a metadata-kinyerésnél ezekre `None`/hiányzó értékkel kell számolni, nem
  feltételezni, hogy mindig ott lesznek.
- **`based on`** ~93%-ban megvan, de van 6 fájl, ahol kettősponttal van írva
  (`based on:` a `based on` helyett) — ez tipikus valós-adat inkonzisztencia,
  amit a szekció-illesztésnél normalizálni kell (pl. kisbetűsítés + kettőspont
  levágása), különben a heading-matching kihagyja ezeket.
- **22 recept (26%)** tartalmaz egy alcímet a főcím alatt, mielőtt az első `##`
  szekció kezdődne — ezt a chunkolásnál a cím részeként, ne külön szekcióként
  érdemes kezelni.
- A fájlhossz nagy szórást mutat (459–4547 karakter) — egy fix karakterszámú
  chunkolásnál (C stratégia) ez azt jelenti, hogy a rövidebb receptek egy chunkba
  férnek, a hosszabbak biztosan több darabra esnek szét.

## 3. Chunking stratégiák

Három stratégiát hasonlítunk össze. Mielőtt belevágnánk B és C stratégiába, egy
technikai megjegyzés: az `unstructured` könyvtár `partition_md()` függvénye a
`spacy`-t importálja a háttérben, ez pedig ezen a gépen egy Windows
alkalmazásvezérlési házirend miatt nem tölthető be (DLL-blokk). Az
`unstructured.chunking.*` és `unstructured.documents.elements` modulok viszont
önmagukban, spacy nélkül is importálhatók — ezért a B/C stratégiánál egy saját,
egyszerű markdown → Element átalakítót írunk, és **a tényleges chunking-logika
(`chunk_elements`, `chunk_by_title`) a valódi `unstructured` könyvtárból jön.**

### Stratégia A: Teljes recept (baseline)

A legegyszerűbb megközelítés: egy recept = egy chunk. Ehhez nincs szükség az
`unstructured` könyvtárra, egyszerűen a teljes fájltartalmat egyben tartjuk meg.

In [7]:
def chunk_strategy_a_full_recipe(filepath: Path) -> dict:
    """Stratégia A: a teljes recept egyetlen chunk."""
    text = filepath.read_text(encoding="utf-8")
    title_match = re.match(r"^#\s+(.+?)\s*$", text, re.M)
    recipe_name = title_match.group(1) if title_match else filepath.stem

    return {
        "chunk_id": f"{filepath.stem}::full",
        "text": text.strip(),
        "metadata": {
            "recipe_name": recipe_name,
            "section_type": "full",
            "source_file": filepath.name,
        },
    }


chunks_a = [chunk_strategy_a_full_recipe(f) for f in files]
print(f"Stratégia A — összesen {len(chunks_a)} chunk (= a receptek száma)")
print()
print("Példa chunk (beef-tacos):")
example = next(c for c in chunks_a if c["metadata"]["source_file"] == "beef-tacos.md")
print(f"  chunk_id: {example['chunk_id']}")
print(f"  metadata: {example['metadata']}")
print(f"  hossz: {len(example['text'])} karakter")
print(f"  szöveg eleje: {example['text'][:120]!r}...")

Stratégia A — összesen 85 chunk (= a receptek száma)

Példa chunk (beef-tacos):
  chunk_id: beef-tacos::full
  metadata: {'recipe_name': 'Beef Tacos', 'section_type': 'full', 'source_file': 'beef-tacos.md'}
  hossz: 560 karakter
  szöveg eleje: '# Beef Tacos  \n\n## info  \n* About 15 minutes  \n* 2 servings  \n\n## ingredients\n* 1 lbs ground beef  \n* 1 tsp ground cumin'...


**Megfigyelés Stratégia A-ról**: egyszerű és megbízható (0 elveszett kontextus egy
recepten belül), de mivel a fájlhossz 459–4547 karakter között szór, a hosszabb
receptek nagy, kevert tartalmú chunkokat adnak — egy „mennyi só kell” típusú
kérdésnél a teljes recept (elkészítési lépésekkel együtt) kerül a modell elé,
ami felesleges kontextust visz be.

### Stratégia B: Szekció alapú

Minden `##` szekció (ingredients, steps, notes, info, based on) külön chunk lesz.
Ehhez két lépés kell:

1. **Saját markdown → Element átalakító** — mivel `partition_md()` nem elérhető
   (lásd fent), a sorokat magunk alakítjuk `unstructured` `Title`/`ListItem`/
   `NarrativeText` objektumokká.
2. **A valódi `unstructured.chunking.title.chunk_by_title()`** csoportosítja az
   elemeket az egyes `Title` (azaz `##` fejléc) köré.

In [8]:
from unstructured.documents.elements import Title, ListItem, NarrativeText
from unstructured.chunking.title import chunk_by_title


def markdown_to_elements(filepath: Path) -> list:
    """Egyszerű, saját markdown-parser: sorokat unstructured Element-ekké alakít.

    A '# ' -> Title (recept címe), '## ' -> Title (szekciócím, ez indít új
    chunk_by_title csoportot), a '* '/'N. ' listaelemek -> ListItem, minden más
    nem üres sor -> NarrativeText.
    """
    text = filepath.read_text(encoding="utf-8")
    elements = []

    for raw_line in text.split("\n"):
        line = raw_line.strip()
        if not line:
            continue

        if line.startswith("# "):
            elements.append(Title(text=line[2:].strip()))
        elif line.startswith("## "):
            elements.append(Title(text=line[3:].strip()))
        elif re.match(r"^[*\-]\s+", line):
            elements.append(ListItem(text=re.sub(r"^[*\-]\s+", "", line)))
        elif re.match(r"^\d+\.\s+", line):
            elements.append(ListItem(text=re.sub(r"^\d+\.\s+", "", line)))
        else:
            elements.append(NarrativeText(text=line))

    return elements


# Demonstráció egy recepten
demo_elements = markdown_to_elements(RECIPES_DIR / "beef-tacos.md")
print(f"beef-tacos.md -> {len(demo_elements)} elem\n")
for el in demo_elements[:8]:
    print(f"  [{type(el).__name__:14s}] {str(el)[:60]}")

beef-tacos.md -> 19 elem

  [Title         ] Beef Tacos
  [Title         ] info
  [ListItem      ] About 15 minutes
  [ListItem      ] 2 servings
  [Title         ] ingredients
  [ListItem      ] 1 lbs ground beef
  [ListItem      ] 1 tsp ground cumin
  [ListItem      ] 1 tsp smoked paprika


In [9]:
def extract_recipe_stats(filepath: Path) -> dict:
    """Recept-szintű metadata, amit minden chunkhoz csatolunk (nem chunk-specifikus)."""
    text = filepath.read_text(encoding="utf-8")

    title_match = re.match(r"^#\s+(.+?)\s*$", text, re.M)
    recipe_name = title_match.group(1) if title_match else filepath.stem

    ing_match = re.search(r"^##\s*ingredients\s*$(.*?)(?=^##|\Z)", text, re.M | re.S)
    ingredients_count = len(re.findall(r"^\s*[*\-]\s+", ing_match.group(1), re.M)) if ing_match else 0

    steps_match = re.search(r"^##\s*steps\s*$(.*?)(?=^##|\Z)", text, re.M | re.S)
    steps_count = len(re.findall(r"^\s*\d+\.\s+", steps_match.group(1), re.M)) if steps_match else 0

    info_match = re.search(r"^##\s*info\s*$(.*?)(?=^##|\Z)", text, re.M | re.S)
    prep_time = None
    if info_match:
        # az info szekció első listaeleme jellemzően az időtartam (pl. "About 15 minutes")
        first_item = re.search(r"^\s*[*\-]\s+(.+?)\s*$", info_match.group(1), re.M)
        if first_item:
            prep_time = first_item.group(1)

    if steps_count <= 3:
        difficulty = "easy"
    elif steps_count <= 7:
        difficulty = "medium"
    else:
        difficulty = "hard"

    return {
        "recipe_name": recipe_name,
        "ingredients_count": ingredients_count,
        "steps_count": steps_count,
        "prep_time": prep_time,
        "difficulty": difficulty,
        "source_file": filepath.name,
    }


def chunk_strategy_b_by_section(filepath: Path) -> list:
    """Stratégia B: szekció-alapú chunking a valódi unstructured chunk_by_title-lal."""
    elements = markdown_to_elements(filepath)
    recipe_stats = extract_recipe_stats(filepath)

    # max_characters nagyra állítva, hogy egy szekció ne törjön szét;
    # combine_text_under_n_chars=0, hogy a rövid szekciók (pl. notes) NE olvadjanak
    # bele a szomszédos szekcióba -> valóban 1 szekció = 1 chunk legyen.
    title_chunks = chunk_by_title(
        elements,
        max_characters=5000,
        combine_text_under_n_chars=0,
        multipage_sections=True,
    )

    chunks = []
    for i, chunk in enumerate(title_chunks):
        chunk_text = str(chunk)
        first_line = chunk_text.split("\n", 1)[0].strip().lower()
        section_type = first_line if first_line in {"ingredients", "steps", "notes", "info", "based on"} else "other"

        chunks.append({
            "chunk_id": f"{filepath.stem}::{section_type}::{i}",
            "text": chunk_text,
            "metadata": {**recipe_stats, "section_type": section_type},
        })
    return chunks


chunks_b = [c for f in files for c in chunk_strategy_b_by_section(f)]
print(f"Stratégia B — összesen {len(chunks_b)} chunk ({len(chunks_b) / len(files):.1f} átlagosan receptenként)")

Stratégia B — összesen 442 chunk (5.2 átlagosan receptenként)


In [10]:
section_type_counts = collections.Counter(c["metadata"]["section_type"] for c in chunks_b)
print("Section type eloszlás:")
for st, cnt in section_type_counts.most_common():
    print(f"  {st:12s} {cnt}")

print()
example_chunks = [c for c in chunks_b if c["metadata"]["source_file"] == "beef-tacos.md"]
print(f"Példa: beef-tacos.md -> {len(example_chunks)} chunk")
for c in example_chunks:
    print(f"\n  chunk_id: {c['chunk_id']}")
    print(f"  metadata: {c['metadata']}")
    print(f"  szöveg: {c['text']!r}")

Section type eloszlás:
  other        93
  ingredients  85
  steps        85
  based on     79
  notes        59
  info         41

Példa: beef-tacos.md -> 5 chunk

  chunk_id: beef-tacos::other::0
  metadata: {'recipe_name': 'Beef Tacos', 'ingredients_count': 8, 'steps_count': 3, 'prep_time': 'About 15 minutes', 'difficulty': 'easy', 'source_file': 'beef-tacos.md', 'section_type': 'other'}
  szöveg: 'Beef Tacos'

  chunk_id: beef-tacos::info::1
  metadata: {'recipe_name': 'Beef Tacos', 'ingredients_count': 8, 'steps_count': 3, 'prep_time': 'About 15 minutes', 'difficulty': 'easy', 'source_file': 'beef-tacos.md', 'section_type': 'info'}
  szöveg: 'info\n\nAbout 15 minutes\n\n2 servings'

  chunk_id: beef-tacos::ingredients::2
  metadata: {'recipe_name': 'Beef Tacos', 'ingredients_count': 8, 'steps_count': 3, 'prep_time': 'About 15 minutes', 'difficulty': 'easy', 'source_file': 'beef-tacos.md', 'section_type': 'ingredients'}
  szöveg: 'ingredients\n\n1 lbs ground beef\n\n1 tsp ground cu

**Megfigyelés Stratégia B-ről**: célzottabb keresést tesz lehetővé (egy
„milyen hozzávalók kellenek” kérdésnél csak az `ingredients` chunkok relevánsak,
nem kell végigolvasni az elkészítést is), viszont **egy chunk elveszíti a recept
nevén kívüli kontextust** — pl. egy önmagában álló `steps` chunkból nem derül ki,
hogy mennyi az adagszám vagy milyen a nehézségi szint, hacsak a metadata nem
pótolja ezt (ezért van minden chunkon `recipe_name`, `ingredients_count`,
`difficulty` is, nem csak a chunk saját szövege). A korábban észlelt
strukturális inkonzisztencia (pl. `bagels.md`-ben egy be nem címkézett második
lépéssor) miatt néhány recept `steps` chunkja hosszabb/összetettebb a
vártnál — ez limitálja, hogy a B stratégia mennyire tekinthető „tisztának”.

Egy konkrét, adatból látszó mellékhatás: az `other` kategória (93 chunk) túlnyomó része a receptek **címsora**, ami saját, önálló 1-soros chunkot kapott (pl. csak `"Beef Tacos"`) — ez azért történik, mert a H1 cím is `Title` elem, a `chunk_by_title` pedig minden `Title`-nál új chunkot nyit. Ez retrieval szempontjából felesleges, alacsony információtartalmú chunk (bár a metadata rajta is teljes) — ez egy tanulság a B stratégia finomításához: a címsort érdemesebb lenne a következő szekcióba (pl. `info` vagy `ingredients`) összeolvasztani, nem külön chunkként kezelni.

### Stratégia C: Karakterszám alapú, overlap-pal

Fix méretű chunkok, a szomszédos chunkok között átfedéssel (overlap), hogy egy
mondat/gondolat ne vágódjon ketté információvesztéssel. Ehhez a valódi
`unstructured.chunking.basic.chunk_elements()` függvényt használjuk — ugyanazokon
az elemeken, amiket a B stratégiánál már előállítottunk (`markdown_to_elements`).

In [11]:
from unstructured.chunking.basic import chunk_elements


def chunk_strategy_c_char_based(filepath: Path, max_characters: int, overlap: int, new_after_n_chars: int = None) -> list:
    """Stratégia C: fix karakterszámú chunkok, overlap-pal."""
    elements = markdown_to_elements(filepath)
    recipe_stats = extract_recipe_stats(filepath)

    char_chunks = chunk_elements(
        elements,
        max_characters=max_characters,
        new_after_n_chars=new_after_n_chars or int(max_characters * 0.8),
        overlap=overlap,
    )

    chunks = []
    for i, chunk in enumerate(char_chunks):
        chunks.append({
            "chunk_id": f"{filepath.stem}::char::{i}",
            "text": str(chunk),
            "metadata": {**recipe_stats, "section_type": "char_based"},
        })
    return chunks


# Alap paraméterezés: 500 karakter, 100 karakter overlap
chunks_c = [c for f in files for c in chunk_strategy_c_char_based(f, max_characters=500, overlap=100)]
print(f"Stratégia C (max_characters=500, overlap=100) — összesen {len(chunks_c)} chunk "
      f"({len(chunks_c) / len(files):.1f} átlagosan receptenként)")

lens_c = [len(c["text"]) for c in chunks_c]
print(f"Chunk-hossz: min={min(lens_c)}, max={max(lens_c)}, átlag={sum(lens_c)//len(lens_c)}")

Stratégia C (max_characters=500, overlap=100) — összesen 313 chunk (3.7 átlagosan receptenként)
Chunk-hossz: min=38, max=496, átlag=381


A feladat kéri a **paraméterekkel való kísérletezést** — nézzük meg, hogyan
változik a chunkok száma és mérete, ha kisebb/nagyobb `max_characters` és
`overlap` értékeket használunk.

In [12]:
param_sets = [
    {"max_characters": 300, "overlap": 50},
    {"max_characters": 500, "overlap": 100},
    {"max_characters": 1000, "overlap": 150},
]

print(f"{'max_characters':>15} {'overlap':>8} {'chunk szám':>11} {'átlag/recept':>13} {'átlag hossz':>12}")
for params in param_sets:
    cs = [c for f in files for c in chunk_strategy_c_char_based(f, **params)]
    lens = [len(c["text"]) for c in cs]
    print(f"{params['max_characters']:>15} {params['overlap']:>8} {len(cs):>11} "
          f"{len(cs) / len(files):>13.1f} {sum(lens)//len(lens):>12}")

 max_characters  overlap  chunk szám  átlag/recept  átlag hossz


            300       50         510           6.0          233


            500      100         313           3.7          381


           1000      150         181           2.1          660


In [13]:
# Példa chunkok beef-tacos.md-ből (max_characters=500, overlap=100)
example_c = [c for c in chunks_c if c["metadata"]["source_file"] == "beef-tacos.md"]
print(f"beef-tacos.md -> {len(example_c)} chunk\n")
for c in example_c:
    print(f"chunk_id: {c['chunk_id']}")
    print(f"hossz: {len(c['text'])} karakter")
    print(f"szöveg: {c['text']!r}")
    print("-" * 60)

beef-tacos.md -> 2 chunk

chunk_id: beef-tacos::char::0
hossz: 429 karakter
szöveg: 'Beef Tacos\n\ninfo\n\nAbout 15 minutes\n\n2 servings\n\ningredients\n\n1 lbs ground beef\n\n1 tsp ground cumin\n\n1 tsp smoked paprika\n\n1/2 tsp garlic powder\n\n1/2 tsp chili powder\n\n1/2 tsp salt\n\n1/2 tsp black pepper\n\n1 tbsp water\n\nsteps\n\nHeat oil in a cast iron skillet over medium heat, add the beef and cook until it starts to brown\n\nAdd seasonings, stir, and cook until done\n\nAdd 1 tbsp water to make the beef a bit saucier, stir to combine'
------------------------------------------------------------
chunk_id: beef-tacos::char::1
hossz: 64 karakter
szöveg: 'based on\n\nhttps://www.howsweeteats.com/2019/01/ground-beef-tacos'
------------------------------------------------------------


**Megfigyelés Stratégia C-ről**: a chunk-szám és a chunk-méret közti kompromisszum
jól látszik a paraméter-táblázatban — kisebb `max_characters` (300) sokkal több,
kisebb chunkot ad (pontosabb találat, de több beágyazás/lekérdezés), nagyobb
(1000) kevesebb, tartalmasabb chunkot, de a recept több, eltérő típusú
információja (hozzávaló + lépés) egy chunkba keveredhet — pont azt a problémát
okozva, amit a B stratégia próbál elkerülni.

Fontos korlát, ami a `beef-tacos.md` példán is látszik: mivel a chunkoló **nem
tudja**, hogy melyik szöveg hozzávaló-lista és melyik elkészítési lépés (csak a
karakterszámot nézi), egy 500 karakteres chunk simán átnyúlhat az `ingredients`
és a `steps` szekció határán — ez az az információvesztés-típus, amit a B
stratégia (szekció-tudatos) kifejezetten elkerül, cserébe a C stratégia
egyenletesebb chunk-méretet ad, ami a vektor-embeddingek szempontjából
kedvezőbb lehet (a nagyon rövid vagy nagyon hosszú chunkok embeddingje
kevésbé megbízható).

### A/B/C stratégiák összehasonlítása

In [14]:
strategies = {
    "A - Teljes recept": chunks_a,
    "B - Szekció alapú": chunks_b,
    "C - Karakter alapú (500/100)": chunks_c,
}

print(f"{'Stratégia':<32} {'Chunk szám':>10} {'Átlag/recept':>13} {'Min hossz':>10} {'Max hossz':>10} {'Átlag hossz':>12}")
print("-" * 92)
for name, chunks in strategies.items():
    lens = [len(c["text"]) for c in chunks]
    print(f"{name:<32} {len(chunks):>10} {len(chunks) / len(files):>13.1f} "
          f"{min(lens):>10} {max(lens):>10} {sum(lens)//len(lens):>12}")

Stratégia                        Chunk szám  Átlag/recept  Min hossz  Max hossz  Átlag hossz
--------------------------------------------------------------------------------------------
A - Teljes recept                        85           1.0        457       4543         1514
B - Szekció alapú                       442           5.2          4       2967          269
C - Karakter alapú (500/100)            313           3.7         38        496          381


| Stratégia | Chunk szám | Erősség | Gyengeség |
|---|---|---|---|
| **A — Teljes recept** | 85 (fix) | Sosem veszik el kontextus egy recepten belül | Hosszú receptnél sok irreleváns szöveg kerül a modell elé |
| **B — Szekció alapú** | 442 | Célzott (csak `ingredients` vagy csak `steps`) | A cím önálló, alacsony értékű chunkká válik; a szekció-metadata nélkül a chunk kontextus nélküli |
| **C — Karakter alapú** | 313 (paraméterfüggő) | Egyenletes chunk-méret (jó az embeddingnek) | Szekcióhatáron keresztül vágja a szöveget, összekeveri a hozzávalókat és a lépéseket |

**Nincs egyértelmű „legjobb”** — ez pontosan az a tanulság, amit a feladat is kér
(„Véleményezés, mi tűnik jónak, mi nem”): A egyszerű és robusztus, de pazarló; B
pontos, de törékeny a strukturális inkonzisztenciákra (lásd `bagels.md`); C
egyenletes, de vak a tartalmi határokra. A gyakorlatban B és C kombinációja
(szekció-tudatos karakterlimit) valószínűleg jobb lenne, mint bármelyik
önmagában — ezt jelen notebookban nem implementáljuk, de érdemes megjegyezni.

### Opcionális: saját logika — ételfajta mint metadata (nem chunkolási stratégia)

Az "ételfajta szerinti darabolás" **nem jó ötlet chunk-határnak**: több teljes
receptet egy chunkba gyömöszölne, ami elmossa az embeddinget és rontja a
találati pontosságot. Ehelyett az ételfajtát (`category`) egy **plusz metadata
mezőként** adjuk minden meglévő (A/B/C) chunkhoz — ez illik a feladat
`filter_by_metadata` funkciójához: a szemantikus keresés kombinálható vele
(„adj csirkés vacsorát” = szemantikus találat + `category=main_dish` szűrés).

A recepteknek nincs explicit kategória-mezőjük a markdownban, ezért egy
**kulcsszó-alapú heurisztikával** következtetünk rá a fájlnévből/címből.

In [15]:
# Sorrend számít: a lista elején lévő szabályok élveznek elsőbbséget,
# hogy pl. a "carnitas-with-salsa-verde" main_dish legyen, ne sauce_condiment
# (a fájlnévben van "salsa", de a fő fogás a carnitas).
CATEGORY_RULES = [
    ("soup", ["soup", "shorba", "pho"]),
    ("salad", ["salad"]),
    ("drink", ["gin-and-tonic", "tonic"]),
    ("bread_dough", ["bread", "tortilla", "pizza", "dough", "pretzel", "roti", "pita", "bagel", "pancake"]),
    ("main_dish", [
        "chicken", "beef", "pork", "turkey", "fish", "tofu", "carnitas", "kebab",
        "kofte", "schnitzel", "nugget", "rib", "enchilada", "taco", "curry",
        "biryani", "carne", "chop",
    ]),
    ("sauce_condiment", [
        "sauce", "salsa", "dressing", "dip", "raita", "mustard", "hummus",
        "guasacaca", "baba-ganoush", "queso",
    ]),
    ("side_dish", [
        "rice", "bean", "potato", "parsnip", "cabbage", "squash", "green-bean",
        "masala", "vada", "samosa", "gobi",
    ]),
]


def classify_dish_category(filename_stem: str) -> str:
    """Kulcsszó-heurisztika a fájlnévből (kötőjelekkel elválasztott szavak)."""
    name = filename_stem.lower()
    for category, keywords in CATEGORY_RULES:
        if any(kw in name for kw in keywords):
            return category
    return "other"


category_by_file = {f.stem: classify_dish_category(f.stem) for f in files}
category_counts = collections.Counter(category_by_file.values())

print("Kategória-eloszlás (kulcsszó-heurisztikával):\n")
for cat, cnt in category_counts.most_common():
    print(f"  {cat:16s} {cnt:3d}")

Kategória-eloszlás (kulcsszó-heurisztikával):

  sauce_condiment   22
  main_dish         21
  side_dish         13
  other             12
  bread_dough       11
  soup               3
  salad              2
  drink              1


In [16]:
# Metadata gazdagítása.
# 1) Az A stratégia chunkjai eddig csak név/szekció/forrásfájl metadatát kaptak; a feladat
#    által kért recept-szintű mezőket (ingredients_count, steps_count, prep_time,
#    difficulty) itt pótoljuk, ahogy a B és C stratégiánál is megvannak.
for c in chunks_a:
    c["metadata"] = {**extract_recipe_stats(RECIPES_DIR / c["metadata"]["source_file"]), "section_type": "full"}

# 2) category hozzáadása minden meglévő (A/B/C) chunkhoz
for chunk_list in (chunks_a, chunks_b, chunks_c):
    for c in chunk_list:
        c["metadata"]["category"] = category_by_file[Path(c["metadata"]["source_file"]).stem]

print("Példák a besorolásra:\n")
for stem in ["beef-tacos", "austro-hungarian-cream-of-horseradish-soup", "hummus",
             "thai-pomelo-salad", "flour-tortillas", "spanish-rice"]:
    print(f"  {stem:45s} -> {category_by_file[stem]}")

Példák a besorolásra:

  beef-tacos                                    -> main_dish
  austro-hungarian-cream-of-horseradish-soup    -> soup
  hummus                                        -> sauce_condiment
  thai-pomelo-salad                             -> salad
  flour-tortillas                               -> bread_dough
  spanish-rice                                  -> side_dish


**Korlátok, amiket érdemes felismerni (nem eltitkolni)**: ez egy **durva
heurisztika**, nem valódi osztályozás — pl. a `channa-masala` és a
`masala-green-beans` is `side_dish`-be esik a "masala" kulcsszó miatt, holott
az előbbi inkább főfogás. Az `egg-biryani` `main_dish` lesz a "biryani"
kulcsszó miatt, ami helyes, de csak mert explicit felvettük a listára — egy
hasonló, de nem listázott étel (`biryani-shorba`, ami leveses biryani-variáns)
a `soup` szabállyal ütközik előbb, és `soup` lesz, nem `main_dish`, mivel a
`soup`-szabály előrébb van a listában. **Egy megbízhatóbb megoldás egy LLM-mel
történő osztályozás lenne** (minden recepthez egy rövid promptot küldve:
„milyen ételfajta ez?”), ami a következő lépésben (LLM-integráció) már
adott — itt egyelőre a heurisztika elég a metadata-szűrés bemutatásához.

## 4. Vector Database (Qdrant)

**Környezeti eltérések az oktató demójától** (mindkettő indokolt, funkcionálisan
egyenértékű helyettesítés):

1. **Docker helyett in-memory Qdrant** (`QdrantClient(":memory:")`) — nincs Docker
   telepítve ezen a gépen. A collection/upsert/search API ugyanaz, mint egy
   valódi szerverrel — csak a kapcsolódási mód más. Hátrány: a feltöltött
   adat **nem marad meg** a kernel újraindítása után (ephemeral).
2. **`fastembed` a `sentence-transformers` helyett** — a `sentence-transformers`
   a `torch`-ra épül, ami ezen a gépen Windows alkalmazásvezérlési házirend
   miatt nem tölthető be (`OSError: [WinError 4551]`, ugyanaz a hiba, mint a
   `spacy`-nál a chunkingnál). A `fastembed` ugyanazokat a nyílt
   sentence-embedding modelleket futtatja, csak ONNX runtime-mal PyTorch
   helyett — ez importálható és működik ezen a gépen.

In [17]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from fastembed import TextEmbedding

# In-memory Qdrant kliens — nincs szükség futó szerverre
qdrant = QdrantClient(":memory:")

# Kis, nyílt embedding modell (384 dimenzió), torch nélkül
embedder = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
EMBEDDING_DIM = 384

print("Qdrant kliens (in-memory) inicializálva")
print(f"Embedding modell: BAAI/bge-small-en-v1.5 ({EMBEDDING_DIM} dimenzió)")

C:\Users\Laci\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qdrant kliens (in-memory) inicializálva
Embedding modell: BAAI/bge-small-en-v1.5 (384 dimenzió)


In [18]:
def upload_chunks_to_collection(chunks: list, collection_name: str):
    """Egy chunk-listát felteszünk egy Qdrant collectionbe, embeddinggel együtt."""
    if qdrant.collection_exists(collection_name):
        qdrant.delete_collection(collection_name)
    qdrant.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

    texts = [c["text"] for c in chunks]
    vectors = list(embedder.embed(texts))

    points = [
        PointStruct(
            id=i,
            vector=vector.tolist(),
            payload={"text": chunk["text"], **chunk["metadata"]},
        )
        for i, (chunk, vector) in enumerate(zip(chunks, vectors))
    ]

    qdrant.upsert(collection_name=collection_name, points=points)
    return len(points)


# Mindhárom stratégiát külön collectionbe töltjük fel, hogy összehasonlíthatók legyenek
collections_map = {
    "recipes_a_full": chunks_a,
    "recipes_b_section": chunks_b,
    "recipes_c_charbased": chunks_c,
}

for name, chunks in collections_map.items():
    n = upload_chunks_to_collection(chunks, name)
    print(f"✓ {name}: {n} pont feltöltve")

✓ recipes_a_full: 85 pont feltöltve


✓ recipes_b_section: 442 pont feltöltve


✓ recipes_c_charbased: 313 pont feltöltve


In [19]:
# Ellenőrzés: mind a 3 collection tényleg létrejött, helyes pontszámmal
print("Collection-ök állapota:\n")
for name in collections_map:
    info = qdrant.get_collection(name)
    print(f"  {name:24s} pontok: {info.points_count:4d}   vektor-dimenzió: {info.config.params.vectors.size}")

Collection-ök állapota:

  recipes_a_full           pontok:   85   vektor-dimenzió: 384
  recipes_b_section        pontok:  442   vektor-dimenzió: 384
  recipes_c_charbased      pontok:  313   vektor-dimenzió: 384


## 5. Keresési funkciók

### `search_recipes` — szemantikus keresés

A kérdést ugyanazzal a `fastembed` modellel alakítjuk számsorrá, amivel a
chunkokat is, majd a Qdrant megkeresi a legközelebbi vektorokat. Alapból a
`recipes_b_section` collectiont használjuk (szekció-alapú), mert ott a
találatok konkrétan megmondják, hogy a recept melyik része (hozzávaló, lépés,
megjegyzés) egyezik a kérdéssel — ez informatívabb, mint a teljes recept.

**A három keresési funkció nem egyformán kapcsolódik a chunkoláshoz.** A `search_recipes` `collection` paramétere bármelyik stratégiára állítható — ezért tudtuk vele korábban A/B/C-t összehasonlítani. A következő két függvény viszont **fixen egy-egy konkrét stratégiához van kötve**, mert a feladatuk csak azzal értelmes: a `search_by_ingredients` a `recipes_b_section`-t használja (csak ott van `ingredients` címkéjű szekció), a `filter_by_metadata` pedig a `recipes_a_full`-t (csak ott van recepenként egyetlen pont, duplikáció nélkül).

In [20]:
def search_recipes(query: str, collection: str = "recipes_b_section", top_k: int = 5):
    """Szemantikus keresés egy Qdrant collectionben."""
    query_vector = list(embedder.embed([query]))[0]

    results = qdrant.query_points(
        collection_name=collection,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    )
    return results.points


def print_search_results(query: str, results, show_text: bool = False):
    print(f"Kérdés: {query!r}")
    print("-" * 70)
    if not results:
        print("  (nincs találat)")
        return
    for i, r in enumerate(results, 1):
        p = r.payload
        section = p.get("section_type", "?")
        print(f"  {i}. [{r.score:.3f}] {p.get('recipe_name', '?'):30s} ({section})")
        if show_text:
            print(f"     {p['text'][:100]!r}...")

In [21]:
# A feladat saját példakérdései
example_queries = [
    "chocolate dessert recipes",
    "quick dinner with chicken",
    "vegetarian meals under 30 minutes",
    "what can I make with flour and eggs?",
    "spicy food recipes",
]

for q in example_queries:
    results = search_recipes(q, top_k=3)
    print_search_results(q, results)
    print()

Kérdés: 'chocolate dessert recipes'
----------------------------------------------------------------------
  1. [0.715] Garlic Bread                   (notes)
  2. [0.690] (Ketchup-based) BBQ Sauce      (notes)
  3. [0.690] Homemade Pasta                 (other)

Kérdés: 'quick dinner with chicken'
----------------------------------------------------------------------
  1. [0.780] Chicken Nuggets                (other)
  2. [0.771] Garlicky Cabbage and Fish Sauce (info)
  3. [0.768] Massaman Curry                 (other)

Kérdés: 'vegetarian meals under 30 minutes'
----------------------------------------------------------------------
  1. [0.779] Dad's Spaghetti Sauce          (info)
  2. [0.777] Vegetarian Chicken Salad       (other)
  3. [0.773] Chicken Schnitzel              (info)

Kérdés: 'what can I make with flour and eggs?'
----------------------------------------------------------------------
  1. [0.791] Homemade Pasta                 (ingredients)
  2. [0.771] Diner-Style P

**Megfigyelés a keresési eredményekről**: a *"what can I make with flour and
eggs?"* és a *"spicy food recipes"* kérdésre releváns találatok jönnek
(Homemade Pasta, Pancakes, ill. csípős ázsiai/mexikói ételek). A *"chocolate
dessert recipes"* kérdésre viszont **semmi releváns nem jön vissza**
(Garlic Bread, BBQ Sauce) — de ez **nem keresési hiba**: a korábbi
kategória-elemzésből tudjuk, hogy a gyűjteményben **0 desszert-recept van**
(`sauce_condiment`, `main_dish`, `bread_dough` dominál). A szemantikus keresés
mindig visszaad valamit (a "legkevésbé távoli" vektort), akkor is, ha semmi
releváns nincs — ez fontos korlát, amire élesben `score_threshold`-dal
érdemes védekezni (a demo notebook is mutat erre példát).

### `search_by_ingredients` — "Mit főzhetek ebből?"

Ez a funkció a `recipes_b_section` collection **`ingredients` chunkjai
között** keres — nem az egész receptben, csak a hozzávaló-listákban —, mert
konkrétan azt kérdezzük, mihez van meg a hozzávaló. Ez a szemantikus keresés
és a metadata-szűrés kombinációja: `section_type == "ingredients"` szűrő +
vektorkeresés.

*Miért fixen `recipes_b_section`?* Ez az egyetlen collection, ahol a `section_type` metadata "ingredients" értéket vehet fel — az A stratégiában minden chunk "full", a C-ben minden "char_based", tehát ott nincs mire szűrni.

In [22]:
from qdrant_client.models import Filter, FieldCondition, MatchValue


def search_by_ingredients(ingredients_list: list, top_k: int = 5):
    """Mit lehet főzni a megadott hozzávalókból? Csak az 'ingredients' chunkok között keres."""
    query_text = "Recipe using: " + ", ".join(ingredients_list)
    query_vector = list(embedder.embed([query_text]))[0]

    results = qdrant.query_points(
        collection_name="recipes_b_section",
        query=query_vector.tolist(),
        query_filter=Filter(
            must=[FieldCondition(key="section_type", match=MatchValue(value="ingredients"))]
        ),
        limit=top_k,
        with_payload=True,
    )
    return results.points


def print_ingredient_results(ingredients_list, results):
    print(f"Hozzávalók: {ingredients_list}")
    print("-" * 70)
    for i, r in enumerate(results, 1):
        p = r.payload
        print(f"  {i}. [{r.score:.3f}] {p['recipe_name']}")
    print()


print_ingredient_results(["flour", "eggs", "butter"], search_by_ingredients(["flour", "eggs", "butter"]))
print_ingredient_results(["tofu", "soy sauce", "garlic"], search_by_ingredients(["tofu", "soy sauce", "garlic"]))
print_ingredient_results(["rice", "chicken"], search_by_ingredients(["rice", "chicken"]))

Hozzávalók: ['flour', 'eggs', 'butter']
----------------------------------------------------------------------
  1. [0.834] Diner-Style Pancakes
  2. [0.834] Homemade Pasta
  3. [0.804] Roti
  4. [0.762] Soft Pretzels
  5. [0.757] Chicken Schnitzel

Hozzávalók: ['tofu', 'soy sauce', 'garlic']
----------------------------------------------------------------------
  1. [0.868] Sesame Tofu
  2. [0.829] Sichuan Three-Pepper Tofu
  3. [0.818] Vegetarian Pho
  4. [0.813] Pad Thai
  5. [0.801] Bonchon-style Sauce

Hozzávalók: ['rice', 'chicken']
----------------------------------------------------------------------
  1. [0.814] Spanish Rice
  2. [0.799] Halal Cart Chicken
  3. [0.758] Pad See Ew
  4. [0.757] Teriyaki Chicken
  5. [0.753] Chicken Schnitzel



### `filter_by_metadata` — szűrés metadata alapján

Ez a függvény **nem használ embeddinget és hasonlítást**: csak a chunkokra
ragasztott címkék (metadata) alapján válogat, mint egy Excel-szűrő. A `recipes_a_full`
collectiont használjuk, mert ott egy pont = egy recept, így nincsenek duplikált
találatok.

Az elkészítési idő (`prep_time`) szabad szöveg (pl. *"About 45 minutes"*,
*"2–4 hours"*, *"about an hour"*), amire nem lehet számként szűrni. Ezért
először egy egyszerű szabállyal kinyerünk belőle egy számot (`prep_minutes`),
és ezt utólag hozzáadjuk a már feltöltött pontok payloadjához (az embeddinget
nem kell újragenerálni).

*Miért fixen `recipes_a_full`?* Mert ott recepenként pontosan egy pont van. Ha a B collectiont használnánk, egy `difficulty="easy"` szűrés ugyanazt a receptet akár 5-ször is visszaadná (a recept mind az 5 chunkja megjelenne), ami egy listázó jellegű kérdésnél félrevezető lenne.

In [23]:
def parse_prep_minutes(text):
    """Az első időmegadást alakítja percre. Tartománynál (pl. '2-4 hours') az alsó határt veszi.
    Ha nincs értelmezhető időadat, None-t ad."""
    if not text:
        return None
    t = text.lower()
    if re.search(r"\ban hour\b", t):
        return 60
    m = re.search(r"(\d+(?:\.\d+)?)(?:\s*[–-]\s*\d+(?:\.\d+)?)?\s*(hours?|hrs?|minutes?|mins?)", t)
    if not m:
        return None
    value, unit = float(m.group(1)), m.group(2)
    return int(value * 60) if unit.startswith("h") else int(value)


# Néhány példa a szabály működésére
for s in ["About 45 minutes", "1.5 hours, 12–24 hours proof, then about 45 minutes",
          "2–4 hours", "about an hour", "About  minutes", None]:
    print(f"  {str(s)!r:60s} -> {parse_prep_minutes(s)}")

  'About 45 minutes'                                           -> 45
  '1.5 hours, 12–24 hours proof, then about 45 minutes'        -> 90
  '2–4 hours'                                                  -> 120
  'about an hour'                                              -> 60
  'About  minutes'                                             -> None
  'None'                                                       -> None


In [24]:
# prep_minutes hozzáadása a recipes_a_full pontokhoz (embedding újragenerálása nélkül)
points_a, _ = qdrant.scroll(collection_name="recipes_a_full", limit=200, with_payload=True)

parsed = 0
for p in points_a:
    minutes = parse_prep_minutes(p.payload.get("prep_time"))
    qdrant.set_payload(collection_name="recipes_a_full", payload={"prep_minutes": minutes}, points=[p.id])
    if minutes is not None:
        parsed += 1

print(f"prep_minutes kinyerve: {parsed}/{len(points_a)} receptnél "
      f"(a többinél nincs időadat az info szekcióban)")

prep_minutes kinyerve: 40/85 receptnél (a többinél nincs időadat az info szekcióban)


In [25]:
from qdrant_client.models import Range


def filter_by_metadata(difficulty=None, category=None, max_prep_minutes=None, limit=100):
    """Szűrés metadata alapján, hasonlítás nélkül. Minden megadott feltételnek teljesülnie kell."""
    conditions = []
    if difficulty:
        conditions.append(FieldCondition(key="difficulty", match=MatchValue(value=difficulty)))
    if category:
        conditions.append(FieldCondition(key="category", match=MatchValue(value=category)))
    if max_prep_minutes is not None:
        conditions.append(FieldCondition(key="prep_minutes", range=Range(lte=max_prep_minutes)))

    points, _ = qdrant.scroll(
        collection_name="recipes_a_full",
        scroll_filter=Filter(must=conditions) if conditions else None,
        limit=limit,
        with_payload=True,
    )
    return [p.payload for p in points]


def show(title, payloads, max_items=8):
    print(f"{title}  ->  {len(payloads)} recept")
    for p in payloads[:max_items]:
        print(f"    - {p['recipe_name']:38s} nehézség={p['difficulty']:6s} kategória={p['category']:16s} idő={p.get('prep_minutes')}")
    if len(payloads) > max_items:
        print(f"    ... (+{len(payloads) - max_items} további)")
    print()


show("difficulty='easy'", filter_by_metadata(difficulty="easy"))
show("category='soup'", filter_by_metadata(category="soup"))
show("max_prep_minutes=30", filter_by_metadata(max_prep_minutes=30))
show("easy + main_dish + max 30 perc", filter_by_metadata(difficulty="easy", category="main_dish", max_prep_minutes=30))

difficulty='easy'  ->  4 recept
    - (Ketchup-based) BBQ Sauce              nehézség=easy   kategória=sauce_condiment  idő=None
    - Beef Tacos                             nehézség=easy   kategória=main_dish        idő=15
    - Bonchon-style Sauce                    nehézség=easy   kategória=sauce_condiment  idő=None
    - Lemon Tahini Dressing                  nehézség=easy   kategória=sauce_condiment  idő=5

category='soup'  ->  3 recept
    - Austro-Hungarian Cream of Horseradish Soup nehézség=hard   kategória=soup             idő=None
    - Biryani Shorba                         nehézség=hard   kategória=soup             idő=90
    - Vegetarian Pho                         nehézség=hard   kategória=soup             idő=60

max_prep_minutes=30  ->  17 recept
    - Beef Tacos                             nehézség=easy   kategória=main_dish        idő=15
    - Black Bean Enchiladas                  nehézség=medium kategória=main_dish        idő=15
    - Burning Bike Hot Sauce         

#### Szűrés + szemantikus keresés együtt

A két módszer kombinálható: a szűrő először leszűkíti a jelölteket, és csak
azok között fut a hasonlóság-keresés. Példa: *"spicy"* keresése csak a
`main_dish` kategórián belül.

In [26]:
query = "spicy food"
query_vector = list(embedder.embed([query]))[0]

for label, flt in [
    ("szűrő nélkül", None),
    ("csak category='main_dish'", Filter(must=[FieldCondition(key="category", match=MatchValue(value="main_dish"))])),
]:
    res = qdrant.query_points(
        collection_name="recipes_a_full",
        query=query_vector.tolist(),
        query_filter=flt,
        limit=4,
        with_payload=True,
    ).points
    print(f"{query!r} - {label}")
    for r in res:
        print(f"    [{r.score:.3f}] {r.payload['recipe_name']:38s} ({r.payload['category']})")
    print()

'spicy food' - szűrő nélkül
    [0.724] Halal Cart Red Sauce                   (sauce_condiment)
    [0.716] Zhug                                   (sauce_condiment)
    [0.715] Pad See Ew                             (other)
    [0.712] Aloo Matar                             (other)

'spicy food' - csak category='main_dish'
    [0.707] Garlicky Cabbage and Fish Sauce        (main_dish)
    [0.690] Thit Heo Nuang Xa                      (main_dish)
    [0.688] Halal Cart Chicken                     (main_dish)
    [0.681] Kai Kaphrao Khai Dao                   (main_dish)



**Megfigyelések a metadata-szűrésről**:

- **Gyors és pontos, de csak annyira jó, mint a metadata.** A `difficulty` a
  lépések számából számolt heurisztika (≤3 easy, ≤7 medium, egyébként hard), a
  `category` a fájlnévből kikövetkeztetett kulcsszó-heurisztika, tehát a
  szűrő ezeknek a hibáit örökli (pl. `biryani-shorba` `soup`-ként szerepel).
- **A `prep_minutes` csak a receptek kb. felénél létezik**, mert az `info`
  szekció nem mindenhol van meg. Az idő-szűrő ezért **csendben kihagyja** azokat
  a recepteket, amiknek nincs időadata, nem azt jelenti, hogy nem elég gyorsak.
  Éles rendszerben ezt jelezni kellene a felhasználónak.
- **Összetett időszövegek** (pl. *"1.5 hours, 12–24 hours proof, then about 45
  minutes"*) esetén az első említett időt vesszük, ami nem az összes idő. Ez
  durva közelítés.
- A szűrés és a szemantikus keresés **kiegészíti egymást**: a szűrő kemény
  igen/nem feltételeket kezel (nehézség, kategória, idő), a szemantikus keresés a
  "jelentés szerinti hasonlóságot" (pl. *"spicy"*), amit szűrővel nem lehet
  kifejezni.

**Konkrét példák a heurisztikák hibáira, amik a fenti kimenetben látszanak:** az *Enchilada Sauce* `main_dish` lett (az `enchilada` kulcsszó a `sauce` előtt van a szabálylistán), a *Garlic Bread* `hard` nehézségű (nyolc+ lépése van, pedig egyszerű étel), és mindössze **4 recept** minősül `easy`-nek, mert a lépésszám-alapú szabály szigorú. A szűrő tehát azt találja meg, amit a címkék mondanak, nem feltétlenül azt, amit egy ember könnyűnek vagy főételnek nevezne.

### A/B/C stratégiák összehasonlítása a keresésnél

Ugyanazt az 5 példakérdést feltesszük mindhárom collectionnek (A: teljes recept,
B: szekció alapú, C: karakter alapú), és összevetjük a top 3 találatot. Így
látjuk, hogy a chunkolási stratégia hogyan hat a keresés minőségére.

In [27]:
compare_collections = [
    ("A - teljes recept", "recipes_a_full"),
    ("B - szekció alapú", "recipes_b_section"),
    ("C - karakter alapú", "recipes_c_charbased"),
]

for q in example_queries:
    print("=" * 78)
    print(f"Kérdés: {q!r}")
    for label, coll in compare_collections:
        res = search_recipes(q, collection=coll, top_k=3)
        print(f"  {label}")
        for r in res:
            p = r.payload
            print(f"     [{r.score:.3f}] {p['recipe_name']:34s} ({p['section_type']})")
    print()

Kérdés: 'chocolate dessert recipes'
  A - teljes recept
     [0.664] Samosa Pie                         (full)
     [0.663] Hasselback Butternut Squash        (full)
     [0.658] Potato and Green Chili Hash        (full)
  B - szekció alapú
     [0.715] Garlic Bread                       (notes)
     [0.690] (Ketchup-based) BBQ Sauce          (notes)
     [0.690] Homemade Pasta                     (other)
  C - karakter alapú
     [0.683] Samosa Pie                         (char_based)
     [0.678] Tarka Dal                          (char_based)
     [0.670] Vegetarian Pho                     (char_based)

Kérdés: 'quick dinner with chicken'
  A - teljes recept
     [0.762] Chicken Nuggets                    (full)
     [0.760] Halal Cart Chicken                 (full)
     [0.751] Chicken Schnitzel                  (full)
  B - szekció alapú
     [0.780] Chicken Nuggets                    (other)
     [0.771] Garlicky Cabbage and Fish Sauce    (info)
     [0.768] Massaman Curry       

  B - szekció alapú
     [0.791] Homemade Pasta                     (ingredients)
     [0.771] Diner-Style Pancakes               (ingredients)
     [0.743] Chicken Schnitzel                  (ingredients)


  C - karakter alapú
     [0.769] Homemade Pasta                     (char_based)
     [0.737] Pad Thai                           (char_based)
     [0.729] Homemade Pasta                     (char_based)

Kérdés: 'spicy food recipes'
  A - teljes recept
     [0.762] Halal Cart Red Sauce               (full)
     [0.752] Garlicky Cabbage and Fish Sauce    (full)
     [0.750] Aloo Matar                         (full)
  B - szekció alapú
     [0.808] Burning Bike Hot Sauce             (steps)
     [0.775] Sichuan Dry-Fried Green Beans      (ingredients)
     [0.768] Aloo Matar                         (ingredients)
  C - karakter alapú
     [0.765] Cucumber Raita                     (char_based)
     [0.762] Gobi Manchurian (Dry)              (char_based)
     [0.760] Egg Biryani                        (char_based)



### Reranking (újrarangsorolás)

A vektorkeresés gyors, de a sorrend csak közelítő. A reranking egy második menet:

1. A Qdrant kihoz **10 jelöltet** (gyors vektorkeresés).
2. Egy **cross-encoder** modell a kérdést és **mind a 10 jelöltet együtt elolvassa**, és pontosabb
   pontszámot ad mindegyiknek.
3. A pontszám szerint újrarendezzük a jelölteket, és az első 3 megy tovább.

Az oktatói demó a `cross-encoder/ms-marco-MiniLM-L-6-v2` modellt használja `sentence-transformers`-szel. Ez
ezen a gépen a `torch` blokkja miatt nem fut, ezért ugyanennek a modellnek az **ONNX-változatát**
(`Xenova/ms-marco-MiniLM-L-6-v2`) használjuk a `fastembed` reranker-moduljával.

In [28]:
from fastembed.rerank.cross_encoder import TextCrossEncoder

reranker = TextCrossEncoder(model_name="Xenova/ms-marco-MiniLM-L-6-v2")


def search_with_rerank(query: str, collection: str, first_k: int = 10, final_k: int = 3):
    """1. menet: vektorkeresés (first_k jelölt). 2. menet: cross-encoder újrarendezés (final_k)."""
    candidates = search_recipes(query, collection=collection, top_k=first_k)
    texts = [c.payload["text"] for c in candidates]
    scores = list(reranker.rerank(query, texts))

    ranked = sorted(zip(candidates, scores), key=lambda pair: pair[1], reverse=True)
    before = [c.payload["recipe_name"] for c in candidates[:final_k]]
    after = [(c, float(s)) for c, s in ranked[:final_k]]
    return before, after


def name_and_section(c):
    return f"{c.payload['recipe_name']} ({c.payload['section_type']})"


#### Hogyan működik a reranking? Lépésről lépésre, egy kérdésen

Ugyanazt a 10 jelöltet nézzük meg **a reranker előtt és után**. Két pontszám van:

- **Vektor-pontszám** (koszinusz-hasonlóság): a kérdés és a chunk **külön-külön** kiszámolt számsorát
  hasonlítja össze. A modell a kérdést és a szöveget sosem látja együtt.
- **Reranker-pontszám**: a cross-encoder a **(kérdés, szöveg) párt együtt olvassa el**, és megmondja, mennyire
  válaszol a szöveg a kérdésre. Pontosabb, de lassabb, ezért csak a 10 jelöltre futtatjuk, nem az összes
  chunkra.

In [29]:
query = "quick dinner with chicken"

# 1. menet: vektorkeresés, 10 jelölt (a sorrend a vektor-pontszám szerinti)
candidates = search_recipes(query, collection="recipes_b_section", top_k=10)
texts = [c.payload["text"] for c in candidates]

# A reranker bemenete: (kérdés, szöveg) párok
print("A reranker bemenete (az első pár):")
print(f"   kérdés: {query!r}")
print(f"   szöveg: {texts[0][:70]!r}...\n")

# 2. menet: a cross-encoder minden párra ad egy pontszámot
rerank_scores = list(reranker.rerank(query, texts))

# Új sorrend a reranker-pontszám szerint
order_after = sorted(range(len(candidates)), key=lambda i: rerank_scores[i], reverse=True)
rank_after = {i: pos + 1 for pos, i in enumerate(order_after)}

print(f"{'előtte':>7} {'vektor':>8} {'reranker':>9} {'utána':>6}   recept (szekció)")
print("-" * 78)
for i, c in enumerate(candidates):
    p = c.payload
    moved = rank_after[i] - (i + 1)
    arrow = "  ^ feljebb" if moved < 0 else ("  v lejjebb" if moved > 0 else "")
    print(f"{i + 1:>7} {c.score:>8.3f} {float(rerank_scores[i]):>9.2f} {rank_after[i]:>6}   "
          f"{p['recipe_name'][:34]} ({p['section_type']}){arrow}")

A reranker bemenete (az első pár):
   kérdés: 'quick dinner with chicken'
   szöveg: 'Chicken Nuggets'...



 előtte   vektor  reranker  utána   recept (szekció)
------------------------------------------------------------------------------
      1    0.780     -6.80      4   Chicken Nuggets (other)  v lejjebb
      2    0.771     -9.90      9   Garlicky Cabbage and Fish Sauce (info)  v lejjebb
      3    0.768     -5.85      3   Massaman Curry (other)
      4    0.765     -7.43      5   Vegetarian Chicken Salad (other)  v lejjebb
      5    0.765     -9.36      7   Aloo Matar (info)  v lejjebb
      6    0.763     -5.40      2   Chicken Schnitzel (other)  ^ feljebb
      7    0.760     -9.67      8   Sichuan Dry-Fried Green Beans (info)  v lejjebb
      8    0.749     -3.79      1   Halal Cart Chicken (steps)  ^ feljebb
      9    0.745     -8.37      6   Larb Gai (other)  ^ feljebb
     10    0.742    -10.58     10   Hasselback Butternut Squash (info)


In [30]:
# Részletes összehasonlítás a B collectionön (ott a legzajosabb a keresés)
for q in example_queries:
    before, after = search_with_rerank(q, "recipes_b_section")
    print("=" * 78)
    print(f"Kérdés: {q!r}   [B - szekció alapú]")
    print("  Reranking előtt (vektorkeresés):")
    for name in before:
        print(f"     - {name}")
    print("  Reranking után:")
    for c, s in after:
        print(f"     - [{s:6.2f}] {name_and_section(c)}")
    print()

Kérdés: 'chocolate dessert recipes'   [B - szekció alapú]
  Reranking előtt (vektorkeresés):
     - Garlic Bread
     - (Ketchup-based) BBQ Sauce
     - Homemade Pasta
  Reranking után:
     - [ -5.39] Enchilada Sauce (notes)
     - [ -6.46] (Ketchup-based) BBQ Sauce (notes)
     - [ -8.16] Garlic Bread (notes)



Kérdés: 'quick dinner with chicken'   [B - szekció alapú]
  Reranking előtt (vektorkeresés):
     - Chicken Nuggets
     - Garlicky Cabbage and Fish Sauce
     - Massaman Curry
  Reranking után:
     - [ -3.79] Halal Cart Chicken (steps)
     - [ -5.40] Chicken Schnitzel (other)
     - [ -5.85] Massaman Curry (other)

Kérdés: 'vegetarian meals under 30 minutes'   [B - szekció alapú]
  Reranking előtt (vektorkeresés):
     - Dad's Spaghetti Sauce
     - Vegetarian Chicken Salad
     - Chicken Schnitzel
  Reranking után:
     - [ -0.79] Kofte Kebabs (info)
     - [ -1.55] Aloo Matar (info)
     - [ -1.82] Milagu Sadam (info)



Kérdés: 'what can I make with flour and eggs?'   [B - szekció alapú]
  Reranking előtt (vektorkeresés):
     - Homemade Pasta
     - Diner-Style Pancakes
     - Chicken Schnitzel
  Reranking után:
     - [  2.05] Homemade Pasta (ingredients)
     - [  0.98] Homemade Pasta (steps)
     - [ -1.35] Diner-Style Pancakes (ingredients)

Kérdés: 'spicy food recipes'   [B - szekció alapú]
  Reranking előtt (vektorkeresés):
     - Burning Bike Hot Sauce
     - Sichuan Dry-Fried Green Beans
     - Aloo Matar
  Reranking után:
     - [  1.61] Burning Bike Hot Sauce (steps)
     - [ -1.07] Spicy Whole-Grain Pub Mustard (other)
     - [ -3.61] Aloo Matar (ingredients)



In [31]:
# Összefoglaló: változott-e az első találat a reranking hatására? (mindhárom collection)
print(f"{'Kérdés':<40} {'Collection':<10} {'1. találat előtte':<30} {'1. találat utána':<30}")
print("-" * 112)
for q in example_queries:
    for label, coll in compare_collections:
        before, after = search_with_rerank(q, coll)
        first_after = after[0][0].payload["recipe_name"]
        changed = "" if before[0] == first_after else "  <- változott"
        print(f"{q[:38]:<40} {label[:1]:<10} {before[0][:28]:<30} {first_after[:28]:<30}{changed}")

Kérdés                                   Collection 1. találat előtte              1. találat utána              
----------------------------------------------------------------------------------------------------------------


chocolate dessert recipes                A          Samosa Pie                     Samosa Pie                    
chocolate dessert recipes                B          Garlic Bread                   Enchilada Sauce                 <- változott
chocolate dessert recipes                C          Samosa Pie                     Cucumber Raita                  <- változott


quick dinner with chicken                A          Chicken Nuggets                Halal Cart Chicken              <- változott


quick dinner with chicken                B          Chicken Nuggets                Halal Cart Chicken              <- változott
quick dinner with chicken                C          Larb Gai                       Halal Cart Chicken              <- változott


vegetarian meals under 30 minutes        A          Vegetarian Chicken Salad       Vegetarian Pho                  <- változott
vegetarian meals under 30 minutes        B          Dad's Spaghetti Sauce          Kofte Kebabs                    <- változott
vegetarian meals under 30 minutes        C          Vegetarian Chicken Salad       Vegetarian Pho                  <- változott


what can I make with flour and eggs?     A          Homemade Pasta                 Homemade Pasta                


what can I make with flour and eggs?     B          Homemade Pasta                 Homemade Pasta                
what can I make with flour and eggs?     C          Homemade Pasta                 Homemade Pasta                


spicy food recipes                       A          Halal Cart Red Sauce           Aloo Matar                      <- változott
spicy food recipes                       B          Burning Bike Hot Sauce         Burning Bike Hot Sauce        


spicy food recipes                       C          Cucumber Raita                 Gobi Manchurian (Dry)           <- változott


**Megfigyelések a rerankingről** (a fenti kimenetek alapján):

- **Javít, ahol a kérdés egyértelmű:** a *"quick dinner with chicken"* kérdésnél mindhárom collectionben a
  *Halal Cart Chicken* kerül az élre, és a B collectionből kiesik a zajos *Garlicky Cabbage and Fish Sauce (info)*
  találat. A *"spicy food"* kérdésnél a C collectionben a hűsítő *Cucumber Raita* helyett a *Gobi Manchurian*
  lesz az első.
- **Ha már jó volt, nem ront:** a *"flour and eggs"* kérdésnél mindenhol a *Homemade Pasta* marad az első.
- **A szám alapú feltételen nem segít, sőt ronthat:** a *"vegetarian meals under 30 minutes"* kérdésnél a B
  collectionben a reranker három `info` chunkot tett előre (*Kofte Kebabs*, *Aloo Matar*, *Milagu Sadam*), mert
  ezekben szerepel egy rövid időadat, de a *Kofte Kebabs* nem vegetáriánus. A reranker a szöveg hasonlóságát
  értékeli, nem számol percet és nem ellenőriz kizáró feltételt. Ehhez metadata-szűrő kell.
- **A pontszám nem használható küszöbnek:** a reranker pontszámai lehetnek negatívak jó találatnál is
  (*Halal Cart Chicken*: -3,79), és a nem releváns *chocolate dessert* találatok is hasonló tartományban vannak
  (-5,39 az első). A pontszám csak **egy kérdésen belüli sorrendhez** hasznos, kérdések között nem
  összehasonlítható.
- **Nincs mit rerankelni, ha a jelöltek között nincs jó:** a *chocolate dessert* kérdésnél a reranker csak a
  rossz találatok között rendez át, desszertet nem tud előhozni.
- **A reranker csak a chunk szövegét látja, a recept nevét nem.** A csak hozzávalókat vagy csak címet tartalmazó
  chunkokat ezért nehezebben ítéli meg.

## 6. Következtetések

### Mit csináltunk

85 markdown receptet háromféleképpen daraboltunk fel (A: teljes recept, 85 chunk; B: szekció alapú, 442 chunk;
C: karakter alapú 500/100, 313 chunk), mindegyikhez metadata-t készítettünk, a chunkokat embeddinggé
alakítottuk (`BAAI/bge-small-en-v1.5`, 384 dimenzió), és három külön Qdrant collectionbe töltöttük.
Három keresési funkciót építettünk: szemantikus keresés, hozzávaló-alapú keresés és metadata-szűrés, valamint egy reranking (újrarangsorolás) lépést.
Ez a RAG folyamat **"R" (retrieval, előkeresés) fele**: a generálás (az LLM válasza a találatok alapján)
nem része a feladatnak.

### Mi működött jól

- **Hozzávaló-alapú keresés** (`search_by_ingredients`): a legjobb eredményt adta. Az `ingredients`
  szekciókra szűkítés kizárja a lépések és jegyzetek zaját, és a hozzávaló-listák jól egyeznek a felsorolt
  hozzávalókkal (pl. tofu + szójaszósz + fokhagyma → Sesame Tofu, Sichuan Three-Pepper Tofu).
- **A stratégia** (teljes recept): tiszta találatok, receptenként egy pont, ismétlés nélkül. Jól működik,
  ha az a kérdés, hogy *melyik receptet* keressük (pl. csirkés vacsora).
- **B stratégia szekció-tudata**: ha a kérdés egy konkrét részre irányul (hozzávalók), a megfelelő
  szekció jön elő (*"flour and eggs"* → Homemade Pasta és Pancakes `ingredients` chunkja).
- **Szűrés és szemantikus keresés kombinálása**: a metadata-szűrő kemény feltételeket kezel (nehézség,
  kategória, idő), a szemantikus keresés a jelentést. A kettő kiegészíti egymást.
- **Reranking (cross-encoder):** ahol a kérdés egyértelmű, javította a sorrendet (*quick dinner with chicken*: mindhárom collectionben a *Halal Cart Chicken* került az élre, és kiesett a zajos *info* találat), és ahol már jó volt az eredmény, nem rontotta el.

### Mi nem működött jól

- **Szám alapú feltételek** (*"vegetarian meals under 30 minutes"*): egyik stratégia sem kezeli, mert az
  embedding a "30 perc" szöveget nem számként értelmezi. Ezt csak a `prep_minutes` metadata-szűrő tudja.
- **Amit nincs a gyűjteményben** (*"chocolate dessert recipes"*): a rendszer így is visszaad valamit, mert
  mindig a "legkevésbé távoli" vektort adja. A gyűjteményben nincs desszert, de a rendszer nem jelzi, hogy nincs
  releváns találat. Éles rendszerben `score_threshold` kellene.
- **B stratégia zaja**: a recept címe és az `info` szekció önálló, nagyon rövid chunk lett (a legrövidebb
  4 karakter). Ezek gyenge, félrevezető találatokat adnak (pl. *Garlicky Cabbage and Fish Sauce (info)*
  egy csirkés kérdésre).
- **C stratégia**: a karakterhatár szekcióhatáron átvág, ezért kevésbé célzott, és ugyanabból a receptből
  duplikált találatot adhat (*Homemade Pasta* kétszer).
- **Becsült metadata**: a `difficulty` (lépésszámból) és a `category` (fájlnévből, kulcsszavakkal)
  heurisztika, nem valódi osztályozás. A szűrés örökli a hibáikat (*Enchilada Sauce* → `main_dish`,
  *Garlic Bread* → `hard`, csak 4 `easy` recept). Az elkészítési idő csak a receptek felénél áll rendelkezésre.
- **Reranking korlátai:** a szám alapú feltételen (*under 30 minutes*) nem segít, a B collectionben ronthat is (a *Kofte Kebabs* nem vegetáriánus), a pontszáma nem használható kérdések közötti küszöbnek, és ha a jelöltek között nincs jó találat (*chocolate dessert*), nem tud jót előhozni.

### Melyik stratégiát választanánk?

| Stratégia | Erősség | Gyengeség | Mikor használnánk |
|---|---|---|---|
| A – teljes recept | tiszta, nincs ismétlés | hosszú recepteknél sok felesleges szöveg | "melyik receptet keressem?" kérdésekhez |
| B – szekció alapú | célzott (pl. csak hozzávalók) | rövid cím/info chunkok zaja | konkrét részre irányuló kérdésekhez |
| C – karakter alapú | egyenletes méret | vak a tartalmi határokra, duplikál | csak akkor, ha nincs használható struktúra |

**Nincs egyértelmű nyertes.** Ennél az adatnál, ahol a struktúra egységes (`ingredients`/`steps`), a **B
stratégia javított változata** volna a legjobb választás.

### Mit csinálnánk másképp

1. **B stratégia finomítása:** a címet és az `info` szekciót olvasztanánk a következő szekcióba, és minden
   chunk elejére beírnánk a recept nevét (pl. *"Beef Tacos - ingredients: ..."*), hogy egy önálló
   `steps` chunk se veszítse el a kontextust.
2. **Hibrid keresés:** a kérdésből kinyernénk a szám alapú és kategória-feltételeket (pl. "under 30 minutes",
   "vegetarian"), és szűrőként alkalmaznánk a szemantikus keresés mellé (ezt egy LLM el tudná végezni).
3. **`score_threshold`** használata, hogy a rendszer jelezze, ha nincs releváns találat.
4. **Metadata LLM-mel:** a `category`, `difficulty` és vegetáriánus jelleg becslését egy LLM-hívásra bíznánk
   kulcsszó-heurisztika helyett.
5. **Generálás:** a reranking utáni találatokból egy LLM fogalmazna meg választ, ezzel lenne teljes a RAG ("G" rész).

### Környezeti eltérések az oktatói demótól (összefoglaló)

| Demóban | Nálunk | Ok |
|---|---|---|
| Docker Qdrant | in-memory Qdrant (`QdrantClient(":memory:")`) | nincs Docker a gépen; az API ugyanaz, az adat nem marad meg |
| `sentence-transformers` (torch) | `fastembed` (ONNX) | a `torch` egy Windows alkalmazásvezérlési házirend miatt nem tölthető be |
| `unstructured` `partition_md()` | saját markdown → Element átalakító + a könyvtár `chunk_by_title` és `chunk_elements` függvényei | a `partition_md()` a `spacy`-t tölti be, amit ugyanaz a házirend blokkol |
| két embedding-modell (Qwen3 és OpenAI) | egy embedding-modell | egyszerűsítés |
| reranking: `cross-encoder/ms-marco-MiniLM-L-6-v2` (`sentence-transformers`) | ugyanez ONNX-változatban: `Xenova/ms-marco-MiniLM-L-6-v2` (`fastembed`) | a `torch` blokkolva van |